In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append(".")

In [3]:
from constrerl.annotator import (
    AnnotatorHelper,
    AnnotatedArticle,
    Entity,
    Relation,
    ConceptDefinition,
    Metadata,
    AnnotationTypes,
    Sentence,
)
from constrerl.sentences import annotated_sentences_to_article, article_to_sentences

RuntimeError: Failed to load shared library '/nfsd/gracedata2/kantz/CLEANR/.venv/lib/python3.12/site-packages/llama_cpp/lib/libllama.so': libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
import pandas as pd


In [ ]:
import llama_cpp as llm

In [ ]:
model = llm.Llama(model_path="quants/hermes-3-1-8B-lora-relations.gguf")

llama_model_load_from_file_impl: using device Metal (Apple M2 Pro) - 21845 MiB free
llama_model_loader: loaded meta data with 36 key-value pairs and 292 tensors from quants/hermes-3-1-8B-lora-relations.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Hermes 3 Llama 3.1 8B
llama_model_loader: - kv   3:                       general.organization str              = NousResearch
llama_model_loader: - kv   4:                           general.basename str              = Hermes-3-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                  

In [ ]:
from pathlib import Path


def load_data(base_path=Path("NEREL-BIO/BioNNE-R/data/en/train/"), collection="train"):
    rel_data = pd.read_table(base_path / f"eng-{collection}-rel.tsv")
    ent_data = pd.read_table(base_path / f"eng-{collection}-ent.tsv")
    articles: dict[str, AnnotatedArticle] = {}
    article_ids = set(rel_data["document_id"]).union(set(ent_data["document_id"]))
    for article_id in article_ids:
        entities = ent_data[ent_data["document_id"] == article_id]
        entities_data: list[Entity] = []
        for _, row in entities.iterrows():
            entities_data.append(
                Entity(
                    start_idx=int(row["entity_span"].split("-")[0]),
                    end_idx=int(row["entity_span"].split("-")[1]),
                    label=row["entity_type"],
                    text_span=row["entity_text"],
                    location="abstract",
                )
            )
        relations = rel_data[rel_data["document_id"] == article_id]
        relations_data: list[Relation] = []
        for _, row in relations.iterrows():
            relations_data.append(
                Relation(
                    predicate=row["relation"],
                    subject_label=row["head_type"],
                    subject_text_span=row["head_text"],
                    subject_start_idx=int(row["head_span"].split("-")[0]),
                    subject_end_idx=int(row["head_span"].split("-")[1]),
                    object_label=row["tail_type"],
                    object_text_span=row["tail_text"],
                    object_start_idx=int(row["tail_span"].split("-")[0]),
                    object_end_idx=int(row["tail_span"].split("-")[1]),
                    subject_location="abstract",
                    object_location="abstract",
                )
            )
        text = ""
        with open(base_path / "texts" / f"{article_id}.txt", "r") as f:
            text = f.read()
        articles[article_id] = AnnotatedArticle(
            entities=entities_data,
            relations=relations_data,
            metadata=Metadata(
                abstract=text,
                title="",
            ),
        )
    return articles


train_articles = load_data()
print(train_articles[train_articles.keys().__iter__().__next__()])

metadata=Metadata(title='', abstract='[Endoprosthetic replacement with lifting of abdominal wall in treatment of umbilical and postoperative ventral hernias].  \nThe results of complex clinical and ultrasonic investigation of abdominal wall and following surgical treatment in 60 women with umbilical and postoperative large ventral hernias combined with abdomen ptosis were analyzed. \nPatients were divided into 2 groups with 30 people per group.  \nEndoprosthetic replacement of abdominal wall defect using standard polypropylene prosthesis was applied in the 1st group, endoprosthetic replacement with musculoaponeurotic tissues lifting in hypogastric area using original super lightweight polypropylenepolyvinylidenefluoride prosthesis--in the 2nd group. \nPolypropylene endoprosthesisconsist of main flap 15×15 cm with roundish edges and additional flap 5×40 cm in the form of wide stripe placed at the lower edge of main flap transversely to its direction. \nIt was revealed increased physical

In [ ]:
# extract ontolgy (entities, relations)

found_relations: set[tuple[str, str, str]]= set()
found_entity_types: set[str] = set()
for article in train_articles.values():
    entities = article.entities
    for ent in entities:
        found_entity_types.add(ent.label)
    for rel in article.relations:
        found_relations.add(
            ( rel.subject_label,rel.predicate, rel.object_label)
        )
        
found_relations = list(found_relations)
found_entity_types = list(found_entity_types)


In [ ]:
annotator = AnnotatorHelper(
    possible_labels=found_entity_types,
    possible_relations=found_relations,
)
annotator.model = model
annotator.load_articles(train_articles)

Embedding articles: 100%|██████████| 55/55 [00:11<00:00,  4.97it/s]


In [ ]:
bionner_out = Path("data/bionner")
annotator.save_articles(bionner_out / "train_processed.json")

In [ ]:
dev_data= load_data(base_path=Path("NEREL-BIO/BioNNE-R/data/en/dev/"), collection="dev")
len(dev_data)

50

In [ ]:
import json

with open(bionner_out / "dev_processed.json", "w") as f:
    json.dump(annotator.loaded_articles, f, indent=4, default=lambda x: x.__dict__)


In [13]:
lm_annotations = annotator.annotate(
    {id: a.metadata for id, a in dev_data.items()}, annotate=[AnnotationTypes.RELATION]
)

Annotating articles:   0%|          | 0/50 [00:00<?, ?it/s]llama_perf_context_print:        load time =    6657.80 ms
llama_perf_context_print: prompt eval time =    6656.73 ms /    47 tokens (  141.63 ms per token,     7.06 tokens per second)
llama_perf_context_print:        eval time =    2757.96 ms /    30 runs   (   91.93 ms per token,    10.88 tokens per second)
llama_perf_context_print:       total time =   10010.89 ms /    77 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 23 prefix-match hit, remaining 33 prompt tokens to eval
llama_perf_context_print:        load time =    6657.80 ms
llama_perf_context_print: prompt eval time =    2548.05 ms /    33 tokens (   77.21 ms per token,    12.95 tokens per second)
llama_perf_context_print:        eval time =    7778.59 ms /    85 runs   (   91.51 ms per token,    10.93 tokens per second)
llama_perf_context_print:       total time =   12348.02 ms /   118 tokens
llama_perf_context_print:    graphs reused 

KeyboardInterrupt: 

In [ ]:
concepts :list[ConceptDefinition] = {}
for article in train_articles.values():
    for entity in article.entities:
        concepts.append(
            ConceptDefinition(
                label=entity.label,
                description=entity.text_span,
            )
        )

In [ ]:
all_entities = [
    entity for article in train_articles.values() for entity in article.entities
]
entity_uniques = {
    entity.text_span: entity
    for article in train_articles.values()
    for entity in article.entities
}
unique_counts = {}
for article in train_articles.values():
    for entity in article.entities:
        if entity.text_span not in unique_counts:
            unique_counts[entity.text_span] = 0
        unique_counts[entity.text_span] += 1
all_relations = [
    relation for article in train_articles.values() for relation in article.relations
]

In [ ]:
import re


def annotate_sentence_entities(
    sentence: Sentence, do_filtering: bool = False
) -> list[Entity]:
    annotated_entities: list[Entity] = []
    for ent in entity_uniques.values():
        name = ent.text_span
        if name in sentence.text:
            if len(name.strip()) <= 2:
                continue
            all_matches = [m for m in re.finditer(rf"{re.escape(name)}", sentence.text)]
            for match in all_matches:
                start_idx = match.start()
                end_idx = match.end() - 1
                annotated_entities.append(
                    Entity(
                        label=ent.label or "DISO",  # default to DDF if no type is given
                        text_span=name,
                        start_idx=start_idx,
                        end_idx=end_idx,
                        location="title" if sentence.title else "abstract",
                    )
                )
    return annotated_entities


def find_relations_in_sentence(
    sentence: Sentence, entities: list[Entity]
) -> list[Relation]:
    found_relations: list[Relation] = []
    # SUBCLASS_OF if one entity is a substring of the other and they are not identical
    for ent in entities:
        for other_ent in entities:
            if (
                ent.start_idx <= other_ent.start_idx
                and ent.end_idx >= other_ent.end_idx
                and ent.text_span != other_ent.text_span
            ):
                found_relations.append(
                    Relation(
                        predicate="SUBCLASS_OF",
                        subject_label=ent.label,
                        subject_text_span=ent.text_span,
                        subject_start_idx=ent.start_idx,
                        subject_end_idx=ent.end_idx,
                        object_label=other_ent.label,
                        object_text_span=other_ent.text_span,
                        object_start_idx=other_ent.start_idx,
                        object_end_idx=other_ent.end_idx,
                        subject_location=ent.location,
                        object_location=other_ent.location,
                    )
                )
    # ABBREVIATION if one entity is an abbreviation of the other (e.g. "inflammatory bowel disease" and "IBD")
    for ent in entities:
        for other_ent in entities:
            ent_words = ent.text_span.split()
            supposed_abbreviation = "".join([w[0].upper() for w in ent_words])
            if supposed_abbreviation == other_ent.text_span.upper() and len(ent.text_span) - len(other_ent.text_span) > 2:
                found_relations.append(
                    Relation(
                        predicate="ABBREVIATION",
                        subject_label=ent.label,
                        subject_text_span=ent.text_span,
                        subject_start_idx=ent.start_idx,
                        subject_end_idx=ent.end_idx,
                        object_label=other_ent.label,
                        object_text_span=other_ent.text_span,
                        object_start_idx=other_ent.start_idx,
                        object_end_idx=other_ent.end_idx,
                        subject_location=ent.location,
                        object_location=other_ent.location,
                    )
                )
    return found_relations


annotated_articles: dict[str, AnnotatedArticle] = {}
for id, article in dev_data.items():
    sentences = article_to_sentences(article.metadata)
    for sentence in sentences:
        entities = annotate_sentence_entities(sentence)
        relations = find_relations_in_sentence(sentence, entities)
        sentence.entities = entities
        sentence.relations = relations
    annotated_articles[id] = annotated_sentences_to_article(
        sentences, article.metadata
    )

In [ ]:
combined_articles:dict[str, AnnotatedArticle] ={}
for id, article in dev_data.items():
    relations = annotated_articles[id].relations + lm_annotations[id].relations
    combined_articles[id] = AnnotatedArticle(
        entities=annotated_articles[id].entities,
        relations=relations,
        metadata=article.metadata,
    )

In [ ]:
# to the original tsv format

# bring back the original format for the relations
output_relations = [
    {
        "document_id": id,
        "relation": relation.predicate,
        "head_text": relation.subject_text_span,
        "head_span": f"{relation.subject_start_idx}-{relation.subject_end_idx}",
        "head_type": relation.subject_label,
        "tail_text": relation.object_text_span,
        "tail_span": f"{relation.object_start_idx}-{relation.object_end_idx}",
        "tail_type": relation.object_label,
    }
    for id, article in combined_articles.items()
    for relation in article.relations
]

pd.DataFrame(output_relations).to_csv(
    bionner_out / "dev_relations.tsv", sep="\t", index=False
)